# Cookbook — worked examples

Eight recipes covering the shapes almost every ULI
indicator takes. Each one runs as-is on the small demo
dataset in `data/demo/`, so you can execute this whole
notebook now, before you have any data of your own.

**You do not need to read this end to end.** Find the
recipe that matches your data — points, polygons, lines,
a raster, a table — copy it, and change the inputs.

| Your data looks like | Recipe |
|---|---|
| Points (shops, clinics, bus stops) | 1 (how many), 2 (how far) |
| Polygons (parks, flood zones, land use) | 3 (how much cover), 4 (which class) |
| Lines (streets, cycle lanes) | 5 |
| A raster (NDVI, temperature, pollution) | 6 |
| A table of values per AGEB or manzana | 7 |
| One number for the whole city | 8 |

After the recipes, section 9 shows the four steps that are
**the same for every indicator**, and section 10 walks one
indicator all the way to a delivered file.

Schema version 1.0.0.

## The demo data

Real Mexicali layers, cut down so they are quick and
committed to the repository — see
[`data/demo/README.md`](../data/demo/README.md) for
provenance. **They are for learning the tooling, not for
producing indicator values.** One column
(`has_sidewalk_FABRICATED`) is invented outright.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import uli

DEMO = os.path.join('..', 'data', 'demo')
GPKG = os.path.join(DEMO, 'demo.gpkg')

ANALYST_DETAILS = {'name': 'Cookbook demo', 'email': None,
                   'institution': None}

print('uli', uli.SCHEMA_VERSION)
print('reference geographies:', uli.geography.available())

In [ ]:
# What is in the demo geopackage?
import pyogrio
for name, kind in pyogrio.list_layers(GPKG):
    layer = gpd.read_file(GPKG, layer=name)
    print(f'{name:18s} {len(layer):>6,} {kind:12s} '
          f'{[c for c in layer.columns if c != "geometry"]}')

### The one thing to understand first

Every recipe produces the **same simple thing**: a table
with two columns, `geo_id` and `value` — one row per unit
of whichever geography you computed on.

```
  geo_id              value
  CONDESA_F001        3.5
  CONDESA_F002        0.0
  ...
```

That is all. Everything after that — the other
geographies, the validation, the file layout — is done for
you by three function calls (section 9).

---
## Recipe 1 — Points → how many, or how dense

**Use this when:** you have point locations and want a count, a count per km², or a count per 1,000 people.

**Indicators like:** access to shops, clinics, bus stops, parks; destination counts; incident counts.

In [ ]:
destinations = gpd.read_file(GPKG, layer='destinations')
shops = destinations[destinations['category'] == 'convenience']
print(f'{len(shops)} convenience stores')

# Three ways to count them.  Pick the one your measure needs.
raw = uli.count_features(shops, 'ageb')
density = uli.count_features(shops, 'ageb', per='sqkm')
per_capita = uli.count_features(shops, 'ageb', per='1000_persons')

pd.DataFrame({
    'geo_id': raw['geo_id'],
    'count': raw['value'],
    'per_sqkm': density['value'].round(2),
    'per_1000_people': per_capita['value'].round(2),
}).sort_values('count', ascending=False).head()

**Watch out:** a count per 1,000 people is `NaN` where nobody lives — that is deliberate, not a bug. Dividing by zero population would give infinity, and the schema forbids it.

---
## Recipe 2 — Points → how far away

**Use this when:** you want the distance to the closest one.

**Indicators like:** the `proximity` lens on any "access to X" indicator.

In [ ]:
markets = destinations[
    destinations['category'] == 'fresh_food_market']

near = uli.distance_to_nearest(markets, 'grid_100m', cap=3000)
near['value'].describe().round(0)

**Watch out:** this is **straight-line** distance, not walking distance along streets. Real walking distance is longer, and the difference is not constant. Say which you used in the measure description. (WP01 uses network distance from the GHSCI pipeline.)

---
## Recipe 3 — Polygons → how much of the unit they cover

**Use this when:** you want a percentage of area: tree canopy, flood extent, park cover, built-up land.

**Indicators like:** vegetation percent, flooding, residential area, public open space coverage.

In [ ]:
open_space = gpd.read_file(GPKG, layer='open_space')

cover = uli.areal_share(open_space, 'ageb', as_percentage=True)
print(cover['value'].describe().round(1))

# A map is the fastest way to see whether it is plausible.
ax = uli.geography.load('ageb').merge(cover, on='geo_id').plot(
    column='value', legend=True, figsize=(9, 5),
    legend_kwds={'label': '% of AGEB area as public open space'})
ax.set_axis_off()

**Watch out:** overlapping polygons are dissolved first, so a park counted twice in your source will not be counted twice here.

---
## Recipe 4 — Polygons → which class dominates

**Use this when:** your polygons carry a category rather than a quantity.

**Indicators like:** land use type, climate zone, hazard category.

In [ ]:
land = gpd.read_file(GPKG, layer='land_classes')
print(land['land_class'].tolist())

classes = uli.dominant_class(land, 'condesa_fraccionamiento',
                             'land_class')
print(classes['value'].value_counts(dropna=False))

**Watch out:** the result is a **label**, not a number, so it cannot go into the index as-is. Usually you want a share instead — "percent residential" — which is Recipe 3 applied to one filtered class. Note too that 7 of the 40 fraccionamientos come back empty: the census layer does not cover them.

---
## Recipe 5 — Lines → how much of the network has something

**Use this when:** your data is an attribute of street segments and you want the share of street length that has it.

**Indicators like:** sidewalk availability, street lighting, cycling infrastructure, level of traffic stress.

In [ ]:
streets = gpd.read_file(GPKG, layer='streets_condesa')
print(streets['highway'].value_counts().head())

# NOTE: has_sidewalk_FABRICATED is invented demo data.
share = uli.network_share(
    streets, 'condesa_fraccionamiento',
    attribute='has_sidewalk_FABRICATED')
share['value'].describe().round(2)

**Watch out:** segments are cut at unit boundaries, so a long road passing through three units is shared between them by length. Units with no streets at all return `NaN`, not zero.

---
## Recipe 6 — A raster → summarise it within each unit

**Use this when:** you have a continuous surface from satellite imagery or a model.

**Indicators like:** NDVI, land surface temperature, air pollutant concentrations, urban heat.

In [ ]:
RASTER = os.path.join(DEMO, 'demo_population_100m.tif')

mean_value = uli.zonal_statistic(RASTER, 'ageb', 'mean')
max_value = uli.zonal_statistic(RASTER, 'ageb', 'max')

pd.DataFrame({
    'geo_id': mean_value['geo_id'],
    'mean': mean_value['value'].round(1),
    'max': max_value['value'].round(1),
}).head()

**Watch out:** this reads the raster once per unit, so it is slow on the 22,355 cells of `grid_100m`. Test on `ageb` first. Also check the raster nodata value is being honoured — a nodata of -9999 averaged into your mean is a silent disaster.

---
## Recipe 7 — A table you already have per unit

**Use this when:** your values already exist for AGEBs or manzanas — from the census, or a spreadsheet a colleague sent.

**Indicators like:** census variables, housing costs, employment, anything administrative.

In [ ]:
census = pd.read_csv(
    os.path.join(DEMO, 'demo_ageb_census.csv'),
    dtype={'CVEGEO': str})

# The only trick: your key must match geo_id exactly.
# For AGEB and manzana, geo_id IS the INEGI CVEGEO.
native = census.rename(
    columns={'CVEGEO': 'geo_id', 'pct_65_plus': 'value'}
)[['geo_id', 'value']]

known = set(uli.geography.units('ageb')['geo_id'])
print(f'{native["geo_id"].isin(known).sum()} of {len(native)} '
      'rows matched a reference AGEB')
native.head()

**Watch out:** if very few rows match, it is almost always a string/number problem — CVEGEO must be read as text or Excel eats the leading zero. That is why `dtype={"CVEGEO": str}` is there.

---
## Recipe 8 — One number for the whole city

**Use this when:** your source only supports a single city-wide figure — a household survey, one air quality station.

**Indicators like:** housing affordability from ENIGH/ENVI, city-level survey measures.

In [ ]:
native = pd.DataFrame({
    'geo_id': ['MX_Mexicali_2025'],
    'value': [42.0],
})

results = uli.harmonise(native, native_scale='city')
results.groupby(['geo_level', 'aggregation_method']).size()

**Watch out:** every finer unit gets the same value, flagged `replicated`. That is honest and it is fine — but the composite index will exclude it at fine scales, because a constant tells you nothing about where in the city is better. Do not try to make it look more detailed than it is.

---
# 9. The part that is always the same

Whichever recipe you used, you now have `native` — a table
of `geo_id` and `value`. Three calls finish the job.

### 9.1 `harmonise` — fill in the other geographies

You computed at one scale. The project needs five. This
does it, using the shared crosswalk, and records honestly
how each value got there.

```python
results = uli.harmonise(native, native_scale='ageb',
                        method='population_weighted_mean')
```

Which `method`?

| Your measure is… | Use |
|---|---|
| something people experience (access, exposure, comfort) | `population_weighted_mean` |
| a property of land (cover, temperature, land use) | `area_weighted_mean` |
| a property of streets | `length_weighted_mean` |
| a count of things | `sum` |

### 9.2 `label` — say which measure this is

```python
results = uli.label(results, meta, 'my_code__quantity')
```

### 9.3 `check` and `write_indicator` — validate and deliver

```python
print(uli.check(results, meta))
uli.write_indicator(results, meta)
```

`write_indicator` **refuses to write** if validation
fails. While you are still working, add
`allow_failure=True` to save a draft anyway.

---
# 10. One indicator, all the way through

Public open space coverage, from raw polygons to a
validated deliverable. This is the whole job.

In [ ]:
# STEP 1 — documentation, pre-filled from the workbook.
meta = uli.metadata_stub(187, analyst=ANALYST_DETAILS)

print('indicator:', meta['indicator']['name_en'])
print('adapted from:', meta['indicator']['adapted_from'][:90])
print()
print('still to fill in:')
for path in uli.todos(meta)[:8]:
    print('  ', path)

In [ ]:
# STEP 2 — the calculation (Recipe 3).
open_space = gpd.read_file(GPKG, layer='open_space')
native = uli.areal_share(open_space, 'grid_100m',
                         as_percentage=True)
native['value'].describe().round(2)

In [ ]:
# STEP 3 — every reporting geography, from that one result.
results = uli.harmonise(
    native,
    native_scale='grid_100m',
    method='area_weighted_mean',   # land cover, not people
)
results.groupby(['geo_level', 'aggregation_method']).agg(
    units=('value', 'size'), with_value=('value', 'count'))

In [ ]:
# STEP 4 — describe the measure, then label the rows.
measure = meta['measures'][0]
measure.update(
    id='access_to_public_open_space__density_percent_cover',
    name_en='Public open space, percent of area',
    description=(
        'Share of each unit covered by public open space '
        'polygons, as a percentage of unit area.'),
    unit='percent',
    value_type='percentage',
    direction='higher_is_better',
    denominator_type='area_sqkm',
    native_scale='grid_100m',
    aggregation_method='area_weighted_mean',
)
meta['measures'] = [measure]

results = uli.label(results, meta, measure['id'])
results.head(3)

In [ ]:
# STEP 5 — validate.  It will fail, and it should: the
# health evidence is still a TODO, and that is the part
# only you can do.
print(uli.check(results, meta))

Read that report from the bottom. The `[ERROR]` lines are
what stops delivery; `[WARN]` lines are worth reading but
will not block you.

The outstanding errors here are the documentation the
project asks of you — an independent health citation, a
named pathway, documented data sources. Fill those in (see
the [analyst guide](../docs/analyst_guide.md) §2) and the
same call passes.

In [ ]:
# STEP 6 — deliver.  Uncomment once validation passes.
# uli.write_indicator(results, meta)
#
# Writes three files under
#   outputs/<work_package>/<indicator_code>/
# and prints the validation report alongside them.

---
# 11. When something goes wrong

| Message | What it means | Fix |
|---|---|---|
| `FileNotFoundError: Reference geographies not found` | `geography/` is missing or you are running from the wrong folder | run the notebook from `notebooks/`; check `uli.geography.available()` |
| `'x' is not a reporting geography` | a typo in a level name | one of `city`, `ageb`, `grid_1000m`, `condesa_fraccionamiento`, `grid_100m`, `manzana`, `condesa_lote` |
| `... is finer than ...; use method='replicated'` | you asked to aggregate *down* | you cannot invent detail; let `harmonise` handle it |
| `measure id ... must start with ...` | `measure_id` does not match `indicator_code` | use `<indicator_code>__<lens>` |
| `N metadata fields still contain a TODO` | the stub is not filled in | `uli.todos(meta)` lists exactly which |
| `only N of 40 Condesa fraccionamientos have a value` | your native scale does not reach Condesa | compute on `grid_100m` if you can |
| Values all `NaN` after a merge | join keys do not match | check `dtype={'CVEGEO': str}` and compare a few ids by eye |
| Everything is zero | usually a CRS mismatch | all project data is EPSG:6366; the helpers reproject for you, so check your **input** |

Still stuck? Bring it to the group with the error message
and the cell that produced it. A question asked early is
cheaper than a week of quiet struggle.